# M2 QLoRA training (Kaggle T4)
Thin runner: all logic lives in `src/training/train_qlora.py`. Settings: GPU T4 x2 (only one is used), Internet ON, Secrets `HF_TOKEN` (write) + `WANDB_API_KEY`.
Run cells in order. Do not start the full run until `--check` and the smoke run pass.

In [ ]:
# 1. Clone an exact commit (push your changes first, then paste the SHA).
COMMIT = "<commit>"
!git clone https://github.com/hlong24082k/Diff-Mind
%cd Diff-Mind
!git checkout $COMMIT

In [ ]:
# 2. Install. unsloth pulls trl/peft/bitsandbytes; wandb is preinstalled on Kaggle.
!pip install -q unsloth
!pip install -q pyyaml pydantic

In [ ]:
# 3. Secrets + single GPU. Must run BEFORE anything imports torch (restart the kernel if you already did).
import os
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
os.environ["HF_TOKEN"] = s.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = s.get_secret("WANDB_API_KEY")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# 4. Pre-flight: masking / EOS / length asserts, then exit. Read the decoded masked split yourself.
!python -m src.training.train_qlora --check

In [ ]:
# 5. Smoke run: 10 steps, checks VRAM + finite loss. It pushes to the Hub: delete those pushes afterwards.
!python -m src.training.train_qlora --max-steps 10 --run-name v1-smoke

In [ ]:
# 6. Full run v1 (~200 steps).
!python -m src.training.train_qlora

In [ ]:
# 7. Record library versions, then paste them into cell 2 as pins (6.2).
!mkdir -p outputs/v1
!pip freeze | grep -iE "unsloth|trl|peft|transformers|bitsandbytes|torch|accelerate" > outputs/v1/versions.txt
!cat outputs/v1/versions.txt